
# Desplegar un modelo de Unity Catalog en un Serving Endpoint con feature lookup

Este notebook muestra un flujo **simple y práctico** para:

1. **Desplegar** un modelo registrado en **Unity Catalog** en un **Model Serving endpoint**.
2. **Consumir** el endpoint desde Python usando un payload en formato `dataframe_records`.
3. **Aprovechar el automatic feature lookup** si el modelo fue empaquetado con **Feature Engineering / Feature Store metadata**.
4. **Detener** el endpoint al final para que no siga consumiendo recursos.

## Cuándo funciona el feature lookup automático

Este patrón funciona cuando se cumplen estas condiciones:

- El modelo fue registrado con `FeatureEngineeringClient.log_model(...)` o `FeatureStoreClient.log_model(...)`.
- Las features necesarias para inferencia ya fueron **publicadas a un online store**.
- En el request incluyes las **lookup keys** que el modelo necesita para recuperar las features.

> Si además quieres **sobrescribir** alguna feature en tiempo de inferencia, puedes incluir esa columna directamente en el payload.



## Parámetros que debes completar

Antes de ejecutar, reemplaza estos valores:

- `CATALOG`, `SCHEMA`, `MODEL_NAME`
- `MODEL_VERSION`
- `ENDPOINT_NAME`
- `REQUEST_ROWS` con las columnas que tu modelo necesita en inferencia

### Qué debe llevar `REQUEST_ROWS`

Como mínimo:

- Las **lookup keys** del modelo, por ejemplo `customer_id`, `product_id`, etc.

Opcionalmente:

- Columnas externas que el modelo requiera y que **no** vienen del feature store.
- Features que quieras **sobrescribir** manualmente en el request.


In [0]:

# Si necesitas instalar/actualizar librerías en un cluster clásico:
# %pip install -U databricks-sdk mlflow requests
# dbutils.library.restartPython()


In [0]:

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput
import mlflow
import json
import requests
import time
from pprint import pprint


In [0]:

# =========================
# 1) Parámetros
# =========================

CATALOG = "mlops_dbx_talk_dev"
SCHEMA = "ezapata"
MODEL_NAME = "telco_churn_model_online"
MODEL_VERSION = "1"   # cambia esto por la versión real del modelo en Unity Catalog

ENDPOINT_NAME = "churn-model-fs-endpoint-demo"
WORKLOAD_SIZE = "Small"
SCALE_TO_ZERO = True

# Nombre completo del modelo en Unity Catalog
MODEL_FULL_NAME = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"

print("Modelo UC:", MODEL_FULL_NAME)
print("Endpoint:", ENDPOINT_NAME)



## Opcional: resolver una versión por alias

Si prefieres desplegar por alias (por ejemplo `Champion`), puedes resolver la versión antes de crear el endpoint:

```python
from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

model_version = client.get_model_version_by_alias(
    name=MODEL_FULL_NAME,
    alias="Champion"
).version
```

En este notebook dejamos la versión fija para mantenerlo sencillo.


In [0]:

# =========================
# 2) Crear cliente SDK
# =========================

w = WorkspaceClient()
mlflow.set_registry_uri("databricks-uc")


In [0]:

# =========================
# 3) Crear endpoint (idempotente)
# =========================

existing_endpoint_names = [e.name for e in w.serving_endpoints.list()]

if ENDPOINT_NAME in existing_endpoint_names:
    print(f"El endpoint '{ENDPOINT_NAME}' ya existe. No se vuelve a crear.")
    endpoint = w.serving_endpoints.get(name=ENDPOINT_NAME)
else:
    print(f"Creando endpoint '{ENDPOINT_NAME}'...")
    endpoint = w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(
            name=ENDPOINT_NAME,
            served_entities=[
                ServedEntityInput(
                    entity_name=MODEL_FULL_NAME,
                    entity_version=MODEL_VERSION,
                    workload_size=WORKLOAD_SIZE,
                    scale_to_zero_enabled=SCALE_TO_ZERO,
                )
            ]
        ),
            )

print("Endpoint listo.")
print(endpoint)


In [0]:

# =========================
# 4) Revisar estado del endpoint
# =========================

endpoint = w.serving_endpoints.get(name=ENDPOINT_NAME)
print("Estado general del endpoint:")
print(endpoint.state)

print("\nConfiguración actual:")
print(endpoint.config)



## Consumo del endpoint con feature lookup

Si el modelo fue empaquetado con metadata de Feature Engineering / Feature Store:

- el endpoint buscará automáticamente las features en el online store,
- y tú solo necesitas enviar las **lookup keys** y cualquier otra columna adicional requerida por el modelo.

### Ejemplo conceptual

Supón que el modelo fue entrenado con features lookup como:

- `customer_id`
- `product_id`

Entonces el payload mínimo podría verse así:

```python
[
    {"customer_id": 1001, "product_id": 501},
    {"customer_id": 1002, "product_id": 777}
]
```

Si quieres sobrescribir una feature específica en inferencia, puedes incluirla también:

```python
[
    {"customer_id": 1001, "product_id": 501, "num_lifetime_purchases": 999}
]
```

En ese caso, Databricks usa tu valor enviado y solo hace lookup de las features faltantes.


In [0]:
FEATURE_TABLE = "mlops_dbx_talk_dev.ezapata.telco_cust_features"

MODEL_FEATURES = ["tenure_months","tenure_years","monthly_charges","total_charges_filled","avg_monthly_charge_lifetime","abs_charges_gap","gender","internet_service","contract_type","payment_method","tenure_bucket","monthly_charge_bucket"]

In [0]:
from pyspark.sql import functions as F
import json

# Tomamos 2 clientes de forma determinística
scoring_df = (
    spark.table(FEATURE_TABLE)
    # .select("customer_id", *MODEL_FEATURES)
    .orderBy(F.rand())
    .limit(2)
)

display(scoring_df)

# Conservamos customer_id aparte para poder mapear predicciones después
customer_ids = [r["customer_id"] for r in scoring_df.select("customer_id").collect()]

# Payload SOLO con las columnas que el modelo usa
payload_records = [
    row.asDict()
    for row in scoring_df.collect()
]

payload = {"dataframe_records": payload_records}

print(json.dumps(payload, indent=2, ensure_ascii=False))

In [0]:
import os
import mlflow.deployments

DATABRICKS_HOST = "https://adb-7405614117683008.8.azuredatabricks.net/"
DATABRICKS_TOKEN = ""
# Ajusta si no los tienes ya definidos en el entorno
os.environ["DATABRICKS_HOST"] = DATABRICKS_HOST
os.environ["DATABRICKS_TOKEN"] = DATABRICKS_TOKEN

client = mlflow.deployments.get_deploy_client("databricks")

response = client.predict(
    endpoint=ENDPOINT_NAME,
    inputs=payload
)

print(response)

In [0]:
preds = response.get("predictions", response)

resultados = list(zip(customer_ids, preds))
for cid, pred in resultados:
    print(f"customer_id={cid} -> prediction={pred}")


## Consumo del endpoint desde una aplicación externa

Fuera de Databricks normalmente se consume por REST con:

- `DATABRICKS_HOST`
- `DATABRICKS_TOKEN` (idealmente OAuth M2M para producción)
- el endpoint `/serving-endpoints/<endpoint>/invocations`

Este ejemplo queda listo para copiar a una app o script externo.


In [0]:
import requests
import json


url = f"{DATABRICKS_HOST}/serving-endpoints/{ENDPOINT_NAME}/invocations"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

resp = requests.post(url, headers=headers, json=payload, timeout=60)
print(resp.status_code)
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))


## Detener el endpoint al final

Para que no quede consumiendo recursos, este notebook hace un **stop** del endpoint.

### Importante

- Esto aplica para endpoints de **custom models**.
- El endpoint no debe tener una actualización en curso.
- El endpoint detenido deja de responder inferencias hasta que lo vuelvas a iniciar.

Si en lugar de detenerlo quieres borrarlo por completo, puedes usar `DELETE /serving-endpoints/{name}` o `w.serving_endpoints.delete(...)`.


In [0]:

# =========================
# 8) Detener el endpoint
# =========================
#
# Usamos REST porque la documentación oficial expone el stop como:
# POST /api/2.0/serving-endpoints/{name}/config:stop

context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
workspace_url = context.apiUrl().get()
api_token = context.apiToken().get()

stop_url = f"{workspace_url}/api/2.0/serving-endpoints/{ENDPOINT_NAME}/config:stop"
headers = {
    "Authorization": f"Bearer {api_token}",
    "Content-Type": "application/json",
}

resp = requests.post(stop_url, headers=headers, timeout=120)

print("Status code stop:", resp.status_code)
if resp.text:
    print(resp.text)
else:
    print("Solicitud de stop enviada.")


In [0]:

# =========================
# 9) Verificar estado después del stop
# =========================

time.sleep(10)

endpoint_after_stop = w.serving_endpoints.get(name=ENDPOINT_NAME)

print("Estado luego del stop:")
print(endpoint_after_stop.state)

print("\nSi el endpoint aún aparece actualizando, espera unos segundos y vuelve a ejecutar esta celda.")



## Resumen

Con este notebook ya tienes un flujo mínimo y reutilizable para:

- desplegar un modelo de Unity Catalog en un endpoint,
- consumirlo por SDK o por REST,
- aprovechar el feature lookup automático,
- y apagar el endpoint al terminar.

Siguientes mejoras naturales:

- desplegar por **alias** (`Champion`),
- agregar validaciones previas del modelo y del endpoint,
- activar **inference tables**,
- o agregar un polling más robusto del estado del stop/start.
